# Chapter 7 · Molecular Dynamics — live notebook

This notebook runs entirely in your browser via the Pyodide kernel.
All state is saved to your browser's local storage; nothing is sent to a server.

Back to the chapter: <https://dongzhaohe321418-lab.github.io/materials-simulation-handbook/ch07-md/>

The defining demonstration of why symplectic integrators win: a harmonic oscillator integrated with Euler vs velocity-Verlet, looking at energy drift over many periods.

Only Pyodide-compatible packages are used (numpy, scipy, matplotlib, ipywidgets).


## 7.1 Euler vs velocity-Verlet on a 1D oscillator

Equation of motion: $\ddot{x} = -x$. Total energy $E = \tfrac{1}{2}(\dot{x}^2 + x^2)$ should be conserved. Euler grows it exponentially; Verlet keeps it bounded forever.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def euler(x0, v0, dt, n_steps):
    x = np.empty(n_steps + 1)
    v = np.empty(n_steps + 1)
    x[0], v[0] = x0, v0
    for n in range(n_steps):
        a = -x[n]
        x[n + 1] = x[n] + dt * v[n]
        v[n + 1] = v[n] + dt * a
    return x, v

def velocity_verlet(x0, v0, dt, n_steps):
    x = np.empty(n_steps + 1)
    v = np.empty(n_steps + 1)
    x[0], v[0] = x0, v0
    a = -x[0]
    for n in range(n_steps):
        x[n + 1] = x[n] + dt * v[n] + 0.5 * dt ** 2 * a
        a_new = -x[n + 1]
        v[n + 1] = v[n] + 0.5 * dt * (a + a_new)
        a = a_new
    return x, v

dt = 0.1
N = 5000
xe, ve = euler(1.0, 0.0, dt, N)
xv, vv = velocity_verlet(1.0, 0.0, dt, N)
E_e = 0.5 * (ve ** 2 + xe ** 2)
E_v = 0.5 * (vv ** 2 + xv ** 2)
t = np.arange(N + 1) * dt

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(t, E_e, label='forward Euler')
ax.semilogy(t, E_v, label='velocity-Verlet')
ax.set_xlabel('time')
ax.set_ylabel('total energy (log scale)')
ax.set_title('Energy conservation: Euler vs Verlet')
ax.legend()
plt.show()


## 7.2 Convergence of Verlet energy fluctuation with step size

Halving $\Delta t$ should quarter the amplitude of the bounded fluctuation — the hallmark of a second-order symplectic scheme.

In [ ]:
import numpy as np

for dt in [0.4, 0.2, 0.1, 0.05]:
    N = int(200 / dt)
    xv, vv = velocity_verlet(1.0, 0.0, dt, N)
    E = 0.5 * (vv ** 2 + xv ** 2)
    print(f'dt = {dt:5.2f}   peak-to-peak dE = {E.max() - E.min():.5f}')
